# Task 3: AI-Driven Natural Language Processing Project

## Exploration and Analysis of a Language Model

This notebook explores the capabilities of a selected language model (LM), tests it on sample text inputs, analyzes context understanding, evaluates language generation quality, and documents strengths, limitations, and improvement areas.

**Selected LM:** `distilgpt2` from Hugging Face Transformers  
**Optional Orchestration Layer:** LangChain `HuggingFacePipeline` wrapper

## 1. LM Selection

For this task, I selected **DistilGPT-2**, a compact GPT-style causal language model. It is suitable for experimentation because it is lightweight compared with larger GPT models, can be used through the open-source Hugging Face ecosystem, and demonstrates important NLP behaviors such as text continuation, prompt sensitivity, fluency, and coherence.

### Why this model?

- It is accessible for student-level experimentation.
- It supports open-ended language generation.
- It can be connected to LangChain for a simple LM application workflow.
- Its smaller size makes limitations easier to observe, especially around reasoning, factuality, and context retention.

### Project alignment

This exploration aligns with NLP and ML advancement goals by studying how language models process prompts, generate responses, and behave under different input scenarios. The notebook also includes ethical considerations such as hallucination risk, bias, and responsible use.

## 2. Research Questions

This notebook answers the following research questions:

1. How well can a compact GPT-style LM generate fluent text?
2. Does the model use context from the prompt when producing a response?
3. How does the model perform across creative, analytical, factual, and safety-sensitive inputs?
4. What are the model's main strengths and limitations?
5. How can prompt design improve the quality of LM output?

## 3. Environment Setup

If the required packages are missing, run the installation command below in a notebook cell or terminal:

```bash
pip install -r ../requirements.txt
```

The first model load may download files from Hugging Face.

In [ ]:
import re
import time
import textwrap
from typing import Dict, List

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 220)

## 4. Model Implementation

The model is loaded through Hugging Face Transformers. The generation parameters are intentionally kept moderate so the responses are varied but not completely uncontrolled.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
    pad_token_id=tokenizer.eos_token_id,
)

### Optional LangChain Wrapper

LangChain can wrap the Hugging Face pipeline and make it easier to connect the LM to larger applications, prompt templates, tools, or retrieval systems. This mirrors the workflow where an application communicates with LangChain, and LangChain routes requests to a model provider such as Hugging Face or OpenAI.

In [ ]:
try:
    from langchain_huggingface import HuggingFacePipeline
except ImportError:
    from langchain_community.llms import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=generator)
type(llm).__name__

## 5. Prompt Scenarios

The prompts below are designed to test multiple LM capabilities: context understanding, summarization-style behavior, factual phrasing, creative generation, reasoning limitations, and safety-aware language.

In [ ]:
scenarios: List[Dict[str, object]] = [
    {
        "scenario": "Context understanding",
        "prompt": "Context: A student named Priya is building an NLP project using a small GPT-style language model. Priya's goal is to test whether the model remembers details from a prompt. Question: What is Priya building? Answer:",
        "expected_keywords": ["NLP", "project", "language", "model"],
    },
    {
        "scenario": "Summarization-style response",
        "prompt": "Summarize this paragraph in one sentence: Language models can generate fluent text, but they may produce incorrect facts, biased wording, or unsupported claims if they are not evaluated carefully. Summary:",
        "expected_keywords": ["language", "models", "incorrect", "evaluate"],
    },
    {
        "scenario": "Creative generation",
        "prompt": "Write a short, friendly introduction for an AI-driven natural language processing project:",
        "expected_keywords": ["AI", "language", "processing", "project"],
    },
    {
        "scenario": "Reasoning limitation",
        "prompt": "Maya has 3 notebooks. She buys 4 more and gives 2 to her friend. How many notebooks does Maya have now? Explain briefly:",
        "expected_keywords": ["5", "notebooks"],
    },
    {
        "scenario": "Ethical and safe response",
        "prompt": "Give neutral advice about using AI tools responsibly in education:",
        "expected_keywords": ["responsibly", "verify", "learning", "privacy"],
    },
]

pd.DataFrame(scenarios)[["scenario", "prompt", "expected_keywords"]]

## 6. Response Generation

The helper function below runs each prompt and captures response text, latency, word count, and a simple keyword match score.

In [ ]:
def clean_generated_text(prompt: str, generated_text: str) -> str:
    """Remove the original prompt from generated text when the pipeline returns prompt + completion."""
    if generated_text.startswith(prompt):
        return generated_text[len(prompt):].strip()
    return generated_text.strip()


def keyword_score(text: str, keywords: List[str]) -> float:
    text_lower = text.lower()
    matches = sum(1 for keyword in keywords if keyword.lower() in text_lower)
    return round(matches / len(keywords), 2) if keywords else 0.0


def lexical_diversity(text: str) -> float:
    tokens = re.findall(r"[A-Za-z']+", text.lower())
    return round(len(set(tokens)) / len(tokens), 2) if tokens else 0.0


def generate_response(prompt: str) -> Dict[str, object]:
    start = time.perf_counter()
    output = generator(prompt, num_return_sequences=1)[0]["generated_text"]
    latency = round(time.perf_counter() - start, 2)
    response = clean_generated_text(prompt, output)
    return {
        "response": response,
        "latency_seconds": latency,
        "word_count": len(response.split()),
        "lexical_diversity": lexical_diversity(response),
    }

In [ ]:
records = []

for item in scenarios:
    result = generate_response(item["prompt"])
    records.append(
        {
            "scenario": item["scenario"],
            "prompt": item["prompt"],
            "response": result["response"],
            "keyword_score": keyword_score(result["response"], item["expected_keywords"]),
            "word_count": result["word_count"],
            "lexical_diversity": result["lexical_diversity"],
            "latency_seconds": result["latency_seconds"],
        }
    )

results_df = pd.DataFrame(records)
results_df

## 7. Quantitative Analysis

The following charts compare response length, keyword coverage, lexical diversity, and latency. These measurements are simple, but they help identify response patterns across prompt types.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

sns.barplot(data=results_df, x="scenario", y="keyword_score", ax=axes[0, 0], color="#5B8DEF")
axes[0, 0].set_title("Keyword Coverage by Scenario")
axes[0, 0].set_ylim(0, 1)

sns.barplot(data=results_df, x="scenario", y="word_count", ax=axes[0, 1], color="#45B39D")
axes[0, 1].set_title("Generated Response Length")

sns.barplot(data=results_df, x="scenario", y="lexical_diversity", ax=axes[1, 0], color="#F4A261")
axes[1, 0].set_title("Lexical Diversity")
axes[1, 0].set_ylim(0, 1)

sns.barplot(data=results_df, x="scenario", y="latency_seconds", ax=axes[1, 1], color="#C77DFF")
axes[1, 1].set_title("Latency in Seconds")

for ax in axes.flat:
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

## 8. Qualitative Analysis

Use the generated outputs to evaluate the model in human terms. The exact text may vary between runs because sampling is enabled.

In [ ]:
for _, row in results_df.iterrows():
    print("=" * 90)
    print(f"Scenario: {row['scenario']}")
    print("Prompt:")
    print(textwrap.fill(row["prompt"], width=100))
    print("\nModel response:")
    print(textwrap.fill(row["response"], width=100))
    print(f"\nKeyword score: {row['keyword_score']} | Words: {row['word_count']} | Latency: {row['latency_seconds']}s")

### Observations

- **Fluency:** The model usually produces grammatically fluent continuations, especially for creative or open-ended prompts.
- **Context use:** The model can sometimes reuse terms from the prompt, but it may not reliably answer direct questions or preserve all details.
- **Reasoning:** Small GPT-style models are not dependable calculators. Even simple arithmetic may be answered incorrectly unless the prompt strongly guides the model.
- **Factuality:** The model can produce confident-sounding unsupported statements, so responses should be verified before real-world use.
- **Safety and ethics:** The model may give broadly useful responsible-use advice, but production systems should include stronger guardrails and human review.

## 9. Prompt Improvement Experiment

Prompt design can improve output quality. The next experiment compares a basic prompt with a more structured prompt.

In [ ]:
basic_prompt = "Explain why language models need evaluation:"

structured_prompt = """
You are writing for a beginner NLP student.
In 3 short bullet points, explain why language models need evaluation.
Mention factuality, bias, and usefulness.
Answer:
""".strip()

comparison = []
for label, prompt in [("Basic", basic_prompt), ("Structured", structured_prompt)]:
    result = generate_response(prompt)
    comparison.append({"prompt_type": label, "prompt": prompt, **result})

comparison_df = pd.DataFrame(comparison)
comparison_df[["prompt_type", "prompt", "response", "word_count", "lexical_diversity", "latency_seconds"]]

### Prompt Experiment Interpretation

A structured prompt gives the model clearer constraints: audience, format, length, and required topics. Even when the model does not follow every instruction perfectly, structured prompting usually produces more focused output than a vague prompt.

## 10. Strengths and Limitations

### Strengths

- Generates fluent and readable text.
- Works well for brainstorming, drafting, and creative continuation.
- Responds to prompt wording and can adapt style.
- Easy to integrate into Python notebooks and LangChain workflows.

### Limitations

- May hallucinate or invent unsupported information.
- Does not guarantee correct reasoning or arithmetic.
- Context window and instruction-following ability are limited compared with larger modern LMs.
- Responses can vary between runs because sampling introduces randomness.
- Ethical risks include bias, overreliance, privacy concerns, and misuse in academic settings.

## 11. Applications and Improvement Areas

### Potential applications

- Educational writing assistants
- Simple chatbot prototypes
- Text completion tools
- Prompt engineering demonstrations
- NLP model evaluation exercises

### Improvement areas

- Use a stronger instruction-tuned model for better task following.
- Add retrieval-augmented generation for factual grounding.
- Include human evaluation rubrics for coherence, correctness, and safety.
- Run multiple trials per prompt to measure consistency.
- Add bias and toxicity checks before deployment.

## 12. Conclusion and Insights

This exploration shows that compact GPT-style language models are useful for demonstrating core NLP generation capabilities. DistilGPT-2 can produce fluent responses and adapt to different prompt styles, making it effective for experimentation and prototypes. However, the analysis also shows important limitations: the model may fail at reasoning, hallucinate facts, and inconsistently follow detailed instructions.

The key insight is that language model performance depends not only on model size, but also on prompt design, evaluation strategy, and responsible deployment practices. For real applications, a stronger model, retrieval support, safety filters, and human review would be necessary.